In [1]:
from pathlib import Path
import numpy as np
import pandas as pd

In [2]:
# compute ratio-based multimodal combination metrics (per-story profiles)
# mci_1 = tanh(MCC / (GrooVist + ε))

EPSILON = 1e-8
ROBERTA_EXCLUDED_STORY_IDS = [1214, 3761, 5047, 5540, 7195, 10499]
gv_dir = Path('./analysis_data/groovist')
mcc_dir = Path('./analysis_data/mcc')
out_dir = Path('./analysis_data/mci')
out_dir.mkdir(parents=True, exist_ok=True)

In [3]:
# build per-story mci tables

df_gv = pd.read_csv(gv_dir / 'groovist_metrics_per_story.csv')
df_gv = df_gv[['model', 'prompt', 'story_id', 'groovist']].copy()
df_gv['story_id'] = df_gv['story_id'].astype(int)

mcc_original_60 = pd.read_csv(mcc_dir / 'mcc_metrics_story_original_60.csv').copy()
mcc_original_60['prompt'] = 'original'

mcc_original_54 = pd.read_csv(mcc_dir / 'mcc_metrics_story_original_54.csv').copy()
mcc_original_54['prompt'] = 'original'

mcc_large_54 = pd.read_csv(mcc_dir / 'mcc_metrics_story_large_54.csv').copy()
mcc_large_54['prompt'] = 'large'

for _df in [mcc_original_60, mcc_original_54, mcc_large_54]:
    _df['story_id'] = _df['story_id'].astype(int)

def build_mci_data(excluded_story_ids=None):
    if excluded_story_ids:
        # Use fully filtered 54-story variants for both prompts.
        df_mcc_story = pd.concat([mcc_original_54, mcc_large_54], ignore_index=True)
    else:
        # Use original_60 plus large_54 in the main table.
        df_mcc_story = pd.concat([mcc_original_60, mcc_large_54], ignore_index=True)

    df_mcc_story = df_mcc_story[['model', 'prompt', 'story_id', 'MCC']].copy()
    df_mci = df_gv.merge(df_mcc_story, on=['model', 'prompt', 'story_id'], how='left')

    if excluded_story_ids:
        excluded_story_ids = {int(x) for x in excluded_story_ids}
        df_mci = df_mci[~df_mci['story_id'].isin(excluded_story_ids)].copy()

    # ratio variants
    df_mci['mci_tanh_mcc_over_gv'] = np.tanh(df_mci['MCC'] / (df_mci['groovist'] + EPSILON))
    df_mci['mci_tanh_gv_over_mcc'] = np.tanh(df_mci['groovist'] / (df_mci['MCC'] + EPSILON))

    return df_mci

In [4]:
df_mci = build_mci_data()
df_mci_54 = build_mci_data(excluded_story_ids=ROBERTA_EXCLUDED_STORY_IDS)

# original, 60
df_mci_original_60 = df_mci[df_mci['prompt'] == 'original'].copy()
mci_agg_original_60 = (
    df_mci_original_60.groupby('model')
    .agg(
        groovist_mean=('groovist', 'mean'),
        MCC_mean=('MCC', 'mean'),
        mci_tanh_mcc_over_gv_mean=('mci_tanh_mcc_over_gv', 'mean'),
        mci_tanh_mcc_over_gv_std=('mci_tanh_mcc_over_gv', 'std'),
        mci_tanh_gv_over_mcc_mean=('mci_tanh_gv_over_mcc', 'mean'),
        count=('story_id', 'count')
    )
    .reset_index()
    .sort_values('model')
)

# original, 54
df_mci_original_54 = df_mci_54[df_mci_54['prompt'] == 'original'].copy()
mci_agg_original_54 = (
    df_mci_original_54.groupby('model')
    .agg(
        groovist_mean=('groovist', 'mean'),
        MCC_mean=('MCC', 'mean'),
        mci_tanh_mcc_over_gv_mean=('mci_tanh_mcc_over_gv', 'mean'),
        mci_tanh_mcc_over_gv_std=('mci_tanh_mcc_over_gv', 'std'),
        mci_tanh_gv_over_mcc_mean=('mci_tanh_gv_over_mcc', 'mean'),
        count=('story_id', 'count')
    )
    .reset_index()
    .sort_values('model')
)

# long, 54
df_mci_large_54 = df_mci_54[df_mci_54['prompt'] == 'large'].copy()
mci_agg_large_54 = (
    df_mci_large_54.groupby('model')
    .agg(
        groovist_mean=('groovist', 'mean'),
        MCC_mean=('MCC', 'mean'),
        mci_tanh_mcc_over_gv_mean=('mci_tanh_mcc_over_gv', 'mean'),
        mci_tanh_mcc_over_gv_std=('mci_tanh_mcc_over_gv', 'std'),
        mci_tanh_gv_over_mcc_mean=('mci_tanh_gv_over_mcc', 'mean'),
        count=('story_id', 'count')
    )
    .reset_index()
    .sort_values('model')
)

display(mci_agg_original_60)

display(mci_agg_original_54)

display(mci_agg_large_54)

,model,groovist_mean,MCC_mean,mci_tanh_mcc_over_gv_mean,mci_tanh_mcc_over_gv_std,mci_tanh_gv_over_mcc_mean,count
0,claude45,0.748818,0.409132,0.458783,0.283520,0.902511,60
1,gpt4o,0.559174,0.441300,0.571752,0.342520,0.805633,60
2,human,0.454289,0.381389,0.446854,0.494653,0.672287,60
3,internvl3,0.595736,0.401864,0.528101,0.336985,0.839451,60
4,llama4scout,0.352517,0.095177,0.148437,0.323359,0.828541,60
5,qwen3vl,0.723372,0.443075,0.492726,0.299250,0.879433,60


,model,groovist_mean,MCC_mean,mci_tanh_mcc_over_gv_mean,mci_tanh_mcc_over_gv_std,mci_tanh_gv_over_mcc_mean,count
0,claude45,0.750251,0.420599,0.470926,0.283718,0.896894,54
1,gpt4o,0.567798,0.458836,0.589369,0.341054,0.794382,54
2,human,0.450898,0.392986,0.457003,0.509055,0.647029,54
3,internvl3,0.596203,0.417650,0.546341,0.337929,0.828429,54
4,llama4scout,0.365369,0.098198,0.150683,0.327169,0.851087,54
5,qwen3vl,0.725016,0.468133,0.518383,0.297885,0.867545,54


,model,groovist_mean,MCC_mean,mci_tanh_mcc_over_gv_mean,mci_tanh_mcc_over_gv_std,mci_tanh_gv_over_mcc_mean,count
0,claude45,0.729018,0.430572,0.487115,0.295120,0.884169,54
1,gpt4o,0.568097,0.493148,0.615783,0.349308,0.768827,54
2,human,0.582645,0.403446,0.553786,0.323461,0.828627,54
3,internvl3,0.595899,0.403484,0.522301,0.333467,0.841716,54
4,llama4scout,0.313420,0.235590,0.358506,0.489665,0.648537,54
5,qwen3vl,0.757569,0.451172,0.493049,0.298047,0.880143,54


In [5]:
from scipy.stats import ttest_rel

MODELS = ['human', 'claude45', 'gpt4o', 'internvl3', 'llama4scout', 'qwen3vl']

def human_model_paired_ttests_mci(df_subset, metric_col='mci_tanh_mcc_over_gv'):
    rows = []
    df_human = df_subset[df_subset['model'] == 'human'].set_index('story_id')

    for model in MODELS:
        if model == 'human':
            continue

        df_model = df_subset[df_subset['model'] == model].set_index('story_id')
        common_ids = sorted(df_human.index.intersection(df_model.index))

        if len(common_ids) < 2:
            rows.append({
                'model': model,
                't_stat': np.nan,
                'p_value': np.nan,
                'n': len(common_ids)
            })
            continue

        paired = pd.DataFrame({
            'human': df_human.loc[common_ids, metric_col],
            'model': df_model.loc[common_ids, metric_col]
        }).dropna()

        if len(paired) < 2:
            rows.append({
                'model': model,
                't_stat': np.nan,
                'p_value': np.nan,
                'n': len(paired)
            })
            continue

        t_stat, p_value = ttest_rel(paired['human'].values, paired['model'].values)

        rows.append({
            'model': model,
            't_stat': float(t_stat),
            'p_value': float(p_value),
            'n': len(paired)
        })

    result = pd.DataFrame(rows).sort_values('model').reset_index(drop=True)
    result['t_stat'] = result['t_stat'].map(lambda x: f"{x:.2f}" if pd.notna(x) else np.nan)
    result['p_value'] = result['p_value'].map(lambda x: f"{x:.3f}" if pd.notna(x) else np.nan)
    return result.set_index('model')[['t_stat', 'p_value', 'n']]


ttest_mci_original_60 = human_model_paired_ttests_mci(df_mci_original_60, metric_col='mci_tanh_mcc_over_gv')
display(ttest_mci_original_60)

ttest_mci_original_54 = human_model_paired_ttests_mci(df_mci_original_54, metric_col='mci_tanh_mcc_over_gv')
display(ttest_mci_original_54)

ttest_mci_large_54 = human_model_paired_ttests_mci(df_mci_large_54, metric_col='mci_tanh_mcc_over_gv')
display(ttest_mci_large_54)

,t_stat,p_value,n
model,,,
claude45,-0.20,0.841,60
gpt4o,-2.13,0.037,60
internvl3,-1.35,0.183,60
llama4scout,4.46,0.000,60
qwen3vl,-0.78,0.440,60


,t_stat,p_value,n
model,,,
claude45,-0.21,0.832,54
gpt4o,-2.04,0.046,54
internvl3,-1.34,0.187,54
llama4scout,4.28,0.000,54
qwen3vl,-0.95,0.345,54


,t_stat,p_value,n
model,,,
claude45,3.37,0.001,54
gpt4o,-3.11,0.003,54
internvl3,1.20,0.236,54
llama4scout,2.90,0.005,54
qwen3vl,2.98,0.004,54


In [6]:
df_mci.to_csv(out_dir / 'mci_ratio_metrics.csv', index=False)
df_mci_54.to_csv(out_dir / 'mci_ratio_metrics_54.csv', index=False)

mci_agg_original_60.to_csv(out_dir / 'mci_ratio_metrics_agg_original_60.csv', index=False)
mci_agg_original_54.to_csv(out_dir / 'mci_ratio_metrics_agg_original_54.csv', index=False)
mci_agg_large_54.to_csv(out_dir / 'mci_ratio_metrics_agg_large_54.csv', index=False)

ttest_mci_original_60.to_csv(out_dir / 'mci_ttest_original_60.csv')
ttest_mci_original_54.to_csv(out_dir / 'mci_ttest_original_54.csv')
ttest_mci_large_54.to_csv(out_dir / 'mci_ttest_large_54.csv')


In [7]:
neg = df_mci[df_mci['mci_tanh_mcc_over_gv'] < 0].copy()
print('negative rows:', len(neg))
display(neg[['model','prompt','story_id','groovist','MCC','mci_tanh_mcc_over_gv']].head())
print('Any negative MCC?', (df_mci['MCC'] < 0).any())
print('All negative rows have negative groovist?', (neg['groovist'] < 0).all() if len(neg) > 0 else True)

negative rows: 5


,model,prompt,story_id,groovist,MCC,mci_tanh_mcc_over_gv
25,human,original,7065,-0.111006,0.712815,-0.999995
26,human,original,6921,-0.055913,0.698150,-1.000000
31,human,original,6111,-0.048480,0.507729,-1.000000
586,llama4scout,large,3230,-0.131174,0.115529,-0.706787
592,llama4scout,large,13596,-0.051533,0.761594,-1.000000


Any negative MCC? False
All negative rows have negative groovist? True
